# Skeleton-EAA-Pose Colab Pipeline

Module 1 filters PKU v1 daily actions. Module 2 is split into Step 2A Detect/Track and Step 2B Pose.


## 1. Mount Drive and Install Repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set this to your repo location in Colab.
REPO_PATH = '/content/Skeleton-EAA-Pose'
%cd {REPO_PATH}

# Base dependencies + YOLO tracking.
!pip install -q -r requirements.txt

# RTMW3D / MMPose stack for Step 2B.
!pip install -q -U openmim
!mim install -q mmengine mmcv mmdet mmpose


## 2. Dataset Paths


In [ ]:
# ?? Ch?n dataset ??????????????????????????????????????????????????????????????
DATASET = 'pku_v1'   # 'pku_v1' | 'pku_v2' | 'tsu'

DRIVE_ROOT = '/content/drive/MyDrive/?ACN-TN_datasets/?ATN/rawdatasets'

if DATASET == 'pku_v1':
    CONFIG_FILE  = f'{REPO_PATH}/configs/pku_v1.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/videos/PKUv1'
    SKELETON_DIR = f'{DRIVE_ROOT}/skeletons/PKU/Skeleton'
    LABEL_DIR    = f'{DRIVE_ROOT}/skeletons/PKU/Label_PKUMMD_v1'
    ACTIONS_XLSX = f'{DRIVE_ROOT}/skeletons/PKU/Actions.xlsx'

    FILTERED_LABEL_DIR    = f'{DRIVE_ROOT}/PKU_MMD_v1/Label_daily'
    FILTERED_ACTIONS_CSV  = f'{DRIVE_ROOT}/PKU_MMD_v1/Actions_daily.csv'
    ACTIONS_FILE          = FILTERED_ACTIONS_CSV
    OUT_DIR               = f'{DRIVE_ROOT}/PKU_MMD_v1/samples_npy'

elif DATASET == 'pku_v2':
    CONFIG_FILE  = f'{REPO_PATH}/configs/pku_v2.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/PKU_MMD_v2/RGB'
    LABEL_DIR    = f'{DRIVE_ROOT}/PKU_MMD_v2/Label'
    ACTIONS_XLSX = f'{DRIVE_ROOT}/PKU_MMD_v2/Actions.xlsx'

    FILTERED_LABEL_DIR    = LABEL_DIR
    ACTIONS_FILE          = ACTIONS_XLSX
    OUT_DIR               = f'{DRIVE_ROOT}/PKU_MMD_v2/samples_npy'

elif DATASET == 'tsu':
    CONFIG_FILE  = f'{REPO_PATH}/configs/tsu.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/TSU/mp4'
    LABEL_DIR    = f'{DRIVE_ROOT}/TSU/Annotation_v1.0'
    OUT_DIR      = f'{DRIVE_ROOT}/TSU/samples_npy'

    FILTERED_LABEL_DIR    = LABEL_DIR
    ACTIONS_FILE          = None

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f'Dataset: {DATASET}, Device: {DEVICE}')
print(f'Video dir: {VIDEO_DIR}')
print(f'Segments dir: {FILTERED_LABEL_DIR}')
print(f'Output dir: {OUT_DIR}')


## 3. Module 1: Filter PKU v1 Daily Actions


In [ ]:
if DATASET == 'pku_v1':
    !python -m eaa_pose.filter_pku_interactions \
        --config "{CONFIG_FILE}" \
        --skeleton-dir "{SKELETON_DIR}" \
        --label-dir "{LABEL_DIR}" \
        --src-actions-xlsx "{ACTIONS_XLSX}" \
        --out-label-dir "{FILTERED_LABEL_DIR}" \
        --out-actions-csv "{FILTERED_ACTIONS_CSV}"
else:
    print('Module 1 filter skipped for', DATASET)


## 4. Step 2A: YOLO26s + ByteTrack Person Tracking


In [ ]:
!python -m eaa_pose.run_tracks \
    --config "{CONFIG_FILE}" \
    --video-dir "{VIDEO_DIR}" \
    --segments-dir "{FILTERED_LABEL_DIR}" \
    --actions-xlsx "{ACTIONS_FILE if ACTIONS_FILE else ''}" \
    --out-dir "{OUT_DIR}" \
    --device "{DEVICE}"


## 5. Inspect Track Stats


In [ ]:
import json
from pathlib import Path

track_stats_path = Path(OUT_DIR) / 'track_stats.json'
track_stats = json.loads(track_stats_path.read_text(encoding='utf-8'))
print(json.dumps(track_stats, indent=2, ensure_ascii=False)[:4000])


## 6. Step 2B: RTMW3D Pose From Track JSON


In [ ]:
!python -m eaa_pose.run_pose \
    --config "{CONFIG_FILE}" \
    --video-dir "{VIDEO_DIR}" \
    --segments-dir "{FILTERED_LABEL_DIR}" \
    --actions-xlsx "{ACTIONS_FILE if ACTIONS_FILE else ''}" \
    --out-dir "{OUT_DIR}" \
    --device "{DEVICE}"


## 7. Inspect Metadata, Pose Stats, and QC Reports


In [ ]:
from pathlib import Path
import json

out_dir = Path(OUT_DIR)
for name in ['metadata.json', 'pose_stats.json']:
    p = out_dir / name
    print('\n====', name, '====')
    print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2, ensure_ascii=False)[:4000])

qc_files = sorted((out_dir / 'qc').glob('*_qc.json'))
print(f'QC reports: {len(qc_files)}')
if qc_files:
    print('First QC file:', qc_files[0])
    print(json.dumps(json.loads(qc_files[0].read_text(encoding='utf-8')), indent=2, ensure_ascii=False)[:4000])


## 8. Verify One SkateFormer Sample


In [ ]:
import glob
import numpy as np

npy_files = sorted(glob.glob(f'{OUT_DIR}/*.npy'))
print('Num samples:', len(npy_files))
if npy_files:
    arr = np.load(npy_files[0])
    print('Sample:', npy_files[0])
    print('Shape:', arr.shape)
    assert arr.ndim == 4
    assert arr.shape[0] == 3
    assert arr.shape[2] == 25
    assert arr.shape[3] == 1
    print('OK: expected SkateFormer layout (3, T, 25, 1)')
